In [1]:
# Experiment (Visualization) note on n-gram language modeling (notebook scale but with visualization)
import torch
from sorl.gat_sim import GAT, GATConfig, BOS_TOKEN_ID
torch.set_float32_matmul_precision('high')  # Enable TF32 for ~2x speedup
# For fast training, set 'BOS_TOKEN_ID' to 15 in 'sorl/gat_sim.py' 

gat_config = GATConfig(
    vocab_sizes=[BOS_TOKEN_ID+1, 16],  # 16 abstract tokens
    n_layer=4,
    n_head=4,
    n_embd=128,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

model = GAT(gat_config)
# model = model.to("cuda")
# model = torch.compile(model)


# 1. load a small subset of the dataset (checked)
# 2. tokenize & build data loader (checked)
# 3. train with SoRL. 
# 4. visualize abstraciton dynamics (similar to copy-n-paste, but more generally on per-n-gram statistics)

In [2]:
from data.tinystory_local import TinyStoriesDataLoader, collect_rollout_statistics, AbstractionStatistics
from data.tinystory_local import visualize_dynamics
import tiktoken 

# TinyStories Dataset + GPT2 tokenizer
# -------------------------------------
num_stories = 100
max_len = 36
K = 4
loader = TinyStoriesDataLoader(num_stories=num_stories, max_len=max_len, chunk_size=K, device='cpu')

doc_len = max_len  + (max_len - 1) // K # <-- doc len contains abstract tokens
batch_size = 8
memory_span = 2 * max_len + 2
attn_blocksize = 1792
max_iterations = 2

# ---- stat collection ---
abs_stats = AbstractionStatistics(
    n_doc=num_stories,
    n_abs=doc_len - max_len,
    abs_vocab_size=model.vocab_sizes[1],
    device=model.device
)

# --- tokenizer --- 
enc = tiktoken.get_encoding("gpt2")
eot = enc._special_tokens['<|endoftext|>']

Loading 100 stories from TinyStories train...
Loaded 100 stories, 3600 tokens total, 0.01 MB
Collected 726 unique 4-chunks


In [ ]:
# train SoRL 

# Dynamic Visualization of SoRL
# 1. select-one SoRL

from sorl.neo_utils import sorl_rollout_v2, select_best_per_doc, select_best_per_doc_v2, sorl_evaluate_v2, compute_vocab_utilization_rate
from sorl.topo import orthogonalize_abs_param
from collections import defaultdict
from sorl.info import SoRLLoss, SoRLLoss_v2, SoRLLoss_v3, SoRLLoss_v4

# --- orthogonal initialization on abs param --- 
orthogonalize_abs_param(model, do_wte=True, do_head=True)

loss_fn = SoRLLoss_v4(model.vocab_sizes[1], decay=0.8, target_vocab_util=0.9)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
n = 2
temperature = torch.tensor([0.0, 5.0], device=model.device)
num_steps = 400 
alpha_abs = 0.1
# alpha_marg_ent = 1.0
alpha_soft_zipf = 1.0
alpha_traj_marg_ent = 0.0
r_min = 1.0 
reward_mode = 1

record = defaultdict(list)
img_frames = []

for step in range(num_steps): 

    optimizer.zero_grad()

    tokens, doc_ids = loader.get_batch(batch_size)

    with torch.no_grad(): 
        # --- breakdown of SoRL search (select one per-document) ---
        search_data, search_ppt = sorl_rollout_v2(tokens, model, n=n, K=K, 
                                    max_iterations=max_iterations,
                                    memory_span=memory_span,
                                    attn_blocksize=attn_blocksize,
                                    temperature=temperature,
                                    truncate_seq_len=False)
        search_ppt = search_ppt.reshape(search_data.shape[0], -1)

        levels = (search_data >= model.vocab_sizes[0]).long()
        best_data, best_ppt, best_ppt_advantage, utility_reward = select_best_per_doc_v2(search_data, search_ppt, levels, r_min=r_min, reward_mode=reward_mode)
        # best_data, best_ppt, best_ppt_advantage = select_best_per_doc(search_data, search_ppt, levels, model)

    # --- compute loss --- 
    traj_loss, abs_loss, zipf_bigram_loss = loss_fn(best_data, model, memory_span, attn_blocksize, utility_reward[1:])
    loss = traj_loss + alpha_abs * abs_loss + alpha_soft_zipf * zipf_bigram_loss

    # --- optimize --- 
    loss.backward() 
    optimizer.step()

    if step % 2 == 0: 
        with torch.no_grad(): 
            temperatures_eval = torch.tensor([0.0, 10.0], device=model.device)
            val_tokens, val_adv, traj_loss, abs_loss, abs_logits, abs_tokens = sorl_evaluate_v2(tokens, model, n=2, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperatures_eval,
                                                                     truncate_seq_len=False)
            _, _, zipf_bigram_loss = loss_fn(val_tokens, model, memory_span, attn_blocksize, torch.ones_like(val_tokens[..., 1:]))
            vocab_util = compute_vocab_utilization_rate(val_tokens, model)
            abs_stats.update(abs_logits, abs_tokens, doc_ids)
            
            record['vocab_util'].append(vocab_util * 100)
            record['search_adv'].append(val_adv.item() * 100)
            record['abs_loss'].append(abs_loss.item())
            record['traj_loss'].append(traj_loss.item())
            record['bigram_rep_rate'].append(abs_stats.bigram_rep_rate)
            # record['kl_soft_zipf'].append(zipf_bigram_loss.item())
            record['marg_cond_ent']

        print(f"\nvalidation step {step} | traj_loss: {traj_loss.item():.2f} | abs_loss: {abs_loss.item():.2f} | search adv: {val_adv.item() * 100:.2f}% | vocab util: {vocab_util * 100:.2f}%  | marg_cond_ent: {zipf_bigram_loss.item():.2f} | bigram rep rate: {abs_stats.bigram_rep_rate:.2f}")
        
        img = visualize_dynamics(abs_stats, loader, model, enc, K, step)
        img_frames.append(img)


validation step 0 | traj_loss: 10.75 | abs_loss: 8.94 | search adv: 0.05% | vocab util: 75.00%  | marg_cond_ent: 61.83 | bigram rep rate: 0.94

validation step 2 | traj_loss: 10.63 | abs_loss: 8.68 | search adv: 0.07% | vocab util: 81.25%  | marg_cond_ent: 30.22 | bigram rep rate: 0.87

validation step 4 | traj_loss: 10.44 | abs_loss: 8.00 | search adv: 0.13% | vocab util: 43.75%  | marg_cond_ent: 15.09 | bigram rep rate: 0.83

validation step 6 | traj_loss: 10.12 | abs_loss: 7.53 | search adv: 0.15% | vocab util: 31.25%  | marg_cond_ent: 7.96 | bigram rep rate: 0.80

validation step 8 | traj_loss: 9.72 | abs_loss: 7.24 | search adv: 0.15% | vocab util: 37.50%  | marg_cond_ent: 4.59 | bigram rep rate: 0.76

validation step 10 | traj_loss: 9.36 | abs_loss: 7.09 | search adv: 0.11% | vocab util: 37.50%  | marg_cond_ent: 2.89 | bigram rep rate: 0.71

validation step 12 | traj_loss: 8.99 | abs_loss: 6.70 | search adv: 0.12% | vocab util: 31.25%  | marg_cond_ent: 2.07 | bigram rep rate: 0.

In [ ]:
# Question #1. What can we do with TinyStories dataset? Besides as a test-bed for SoRL? 
# -> we can check per-seq embedding

# (a). Compositional Learning
# -> (1). Can SoRL avoids catastrophic forgetting? 
# -> (2). Can SoRL achieves compositional generalization? 

In [25]:
# Generate with SoRL trained on TinyStories Dataset
# -------------------------------------------------

from sorl.neo_utils import * 
def generate(model, idx, K, max_iterations=0, memory_span=1792, attn_blocksize=1792, temperature=0.0):
    
    # --- insert placeholder tokens ---
    insert_mask = infer_rythmic_insert_mask(idx, K, model.vocab_sizes[0])
    idx = insert_tokens(idx, insert_mask, model.vocab_sizes[0].item())

    # --- prefix recursion (abstraction) ---
    # recursion_mask = (idx >= model.vocab_sizes[0]) # abstract recursion
    recursion_mask = (idx == model.vocab_sizes[0]) # abstract pre-fill only
    recursion_mask[:, 0] = False
    for _ in range(max_iterations): 
        _, logits = model.forward(idx, memory_span, attn_blocksize)
        idx = extract_and_sample(
            logits, idx, recursion_mask, model.vocab_sizes, temperature
        )
    
    # --- generate next trajectory token ---
    _, logits = model.forward(idx, memory_span, attn_blocksize)
    next_token_logits = logits[:, -1, :model.vocab_sizes[0]]

    if temperature == 0.0:
        new_tokens = torch.argmax(next_token_logits, dim=-1)
    else:
        probs = F.softmax(next_token_logits / temperature, dim=-1)
        new_tokens = torch.multinomial(probs, num_samples=1).squeeze(-1)

    idx = torch.cat((idx, new_tokens.unsqueeze(1)), dim=1)
    return idx

min_temperature = 0.0
tokens, doc_ids = loader.get_batch(1)

idx = tokens[:, :15].clone()

for i in range(30): 
    idx = generate(model, idx, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=1792, temperature=torch.tensor(min_temperature))
    idx_without_abstraction = idx[idx < model.vocab_sizes[0]]
    # print(f"step {i+1} idx (abstraction free): {idx_without_abstraction.tolist()}")
    # print(f"                         idx : {idx[0].tolist()}")

# correct_cp = torch.allclose(idx_without_abstraction[1 : 1 + seq_len], idx_without_abstraction[1 + seq_len : 2 + 2*seq_len])
# print(f"Copy correct: {correct_cp}")

In [41]:
text = "Once upon a time, "
idx = torch.tensor(enc.encode(text)).unsqueeze(0)
for i in range(30): 
    idx = generate(model, idx, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=1792, temperature=torch.tensor(min_temperature))
    idx_without_abstraction = idx[idx < model.vocab_sizes[0]]

In [ ]:
traj_idx = idx[idx < model.vocab_sizes[0]]
abs_idx = idx[idx >= model.vocab_sizes[0]]

visualize_stream(traj_idx)

In [40]:

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import io
from PIL import Image

def visualize_stream_thought(history, enc, model, K=4, max_steps=10):
    """
    Visualizes the streaming generation process.
    
    Args:
        history: List of (text_tokens_tensor, abs_tokens_tensor) tuples.
                 Each entry is a snapshot of the sequence at a generation step.
        enc: Tokenizer
        model: Model (for vocab sizes)
        K: Chunk size (abstraction ratio)
        max_steps: How many recent steps to show.
    """
    
    steps_to_show = history[-max_steps:]
    n_steps = len(steps_to_show)
    
    fig_height = 1.2 * n_steps
    fig, axes = plt.subplots(n_steps, 1, figsize=(12, fig_height))
    if n_steps == 1: axes = [axes]
    
    abs_start = model.vocab_sizes[0]
    tab20 = plt.cm.tab20.colors

    for i, (tokens, abs_tokens) in enumerate(steps_to_show):
        ax = axes[i]
        ax.set_xlim(0, 12)
        ax.set_ylim(0, 2)
        ax.axis('off')
        
        # Data prep
        # Filter out padding/BOS if needed, or assume raw sequence
        flat_tokens = tokens.flatten().tolist()
        flat_abs = abs_tokens.flatten().tolist()
        
        # Remove placeholders from abs
        valid_abs = [a for a in flat_abs if a != abs_start and a != 0]
        # Normalize abs IDs
        valid_abs_norm = [a - abs_start if a >= abs_start else a for a in valid_abs]
        
        n_chunks = len(flat_tokens) // K
        
        # --- Draw Abstraction (Top) ---
        abs_x_start = 0.5
        abs_spacing = 0.8
        
        for j, a_id in enumerate(valid_abs_norm):
            color = tab20[a_id % 20]
            
            # Highlight if this is the NEWEST abstraction and we are at the step where it appeared
            # Simple heuristic: last one in the list
            is_new = (j == len(valid_abs_norm) - 1) and (i == n_steps - 1) 
            
            # Draw Box
            rect = FancyBboxPatch((abs_x_start + j*abs_spacing, 1.2), 0.6, 0.5,
                                  boxstyle="round,pad=0.1",
                                  facecolor=color, alpha=0.9 if is_new else 0.4,
                                  edgecolor='red' if is_new else 'black',
                                  linewidth=2 if is_new else 1)
            ax.add_patch(rect)
            
            ax.text(abs_x_start + j*abs_spacing + 0.3, 1.45, f"a{a_id}", 
                    ha='center', va='center', weight='bold', fontsize=12)

        # --- Draw Text (Bottom) ---
        # We group text by chunks to align with abstractions
        
        text_x_start = 0.5
        
        for j in range(n_chunks + 1): # +1 for partial chunk
            chunk_tokens = flat_tokens[j*K : (j+1)*K]
            if not chunk_tokens: continue
            
            chunk_text = enc.decode(chunk_tokens)
            
            # Align with abstraction j
            x_pos = text_x_start + j*abs_spacing
            
            # Draw Connection Arrow
            if j < len(valid_abs_norm):
                 ax.annotate("", xy=(x_pos + 0.3, 0.8), xytext=(x_pos + 0.3, 1.2),
                            arrowprops=dict(arrowstyle="->", color="gray", alpha=0.5))

            # Draw Text
            # Highlight newest text
            is_new_text = (j == n_chunks) # Partial chunk at the end
            
            ax.text(x_pos + 0.3, 0.5, chunk_text, 
                    ha='center', va='top', 
                    fontsize=10, 
                    family='monospace',
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.8))

        # Step Label
        ax.text(0, 1.0, f"Step {len(history) - n_steps + i + 1}", 
                fontsize=14, weight='bold', color='#333', rotation=90, va='center')

    plt.tight_layout()
    
    # Convert to Image
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0); img = Image.open(buf).copy(); buf.close(); plt.close(fig)
    return img

In [ ]:
model.vocab_sizes[0]
torch.log(model.vocab_sizes[0])

# traj_marg_ent = 1.0, bigran zipf reg: 1.0
# traj_loss: 1.51 | abs_loss: 0.46 | search adv: 18.40% | vocab util: 75.00%  | kl_soft_zipf: 0.15 | bigram rep rate: 0.00 | traj_marg_ent: 1.43
# degrades search adv, this is not a good idea, drop it
 

# Question #1. how can H(P(s)) maxed out whilst having high p(s | a)? 
#              is it a bug in my implementation (how I extract those logits?)



tensor(10.8249)

In [1]:
# Request #1. 
# -> build experiement pipeline to test out zipf regularization on FineWeb & TinyStories dataset

# Idea #1. 
# -> separatibility (contrast) can be induced by regularizing on MBE for abstract representations (WTE)
# -> I wonder what effect does this has -- can it enlarge search adv? 

# Idea #2. 
# -> an extra scalar reward r = max(p(s|a) - p(s), r_min) can create an extra 'rich gets richer' dynamics
#    that helps improve search adv?

# Idea #3. 
# -> explicitly regularize on p(s), try to push it up to see if we can further improve 'search advantage'


In [ ]:
# marg cond w = 1.0 
# traj_loss: 0.85 | abs_loss: 0.04 | search adv: 21.01% | vocab util: 93.75%  | marg_ent: 2.67 | cond_ent: 0.05 | avg cos sim: 0.44

# ce zipf w = 1.0 
# traj_loss: 0.90 | abs_loss: 0.01 | search adv: 26.56% | vocab util: 6.25% (collapsed) | ce_zipf: 1.18 | ce_soft_zipf: 3.26

# kl zip w = 1.0 
# traj_loss: 1.21 | abs_loss: 0.28 | search adv: 31.68% | vocab util: 37.50%  | kl_zipf: 0.05 | kl_soft_zipf: 1.81  | avg cos sim: 0.80
# (traj loss is still dropping for the record)

# soft kl zip w = 1.0
# traj_loss: 0.87 | abs_loss: 0.51 | search adv: 30.31% | vocab util: 37.5% - 56%  | kl_zipf: 0.04 | kl_soft_zipf: 0.12 | avg cos sim: 0.87
# -> soft kl seems to be a better target

# soft bigram zipf kl w = 1.0 
# traj_loss: 0.89 | abs_loss: 0.53 | search adv: 25.84% | vocab util: 68.75%  | kl_soft_zipf: 0.24 | avg cos sim: 0.47
# -> visually I observe much less 'repetitions'

# soft bigram zipf kl w = 1.0 & utility reward scaling r = max(p(s|a)/p(s), 1.0)
# traj_loss: 0.79 | abs_loss: 0.40 | search adv: 30.13% | vocab util: 62.50%  | kl_soft_zipf: 0.17 | avg cos sim: 0.44

# marg ent reg w = 1.0 & utility reward scaling r = max(p(s|a)/p(s), 1.0)
# traj_loss: 0.56 | abs_loss: 0.01 | search adv: 32.70% | vocab util: 6.25%  | marg_cond_ent: 0.01 | bigram rep rate: 0.99
# ==> vocabulary collapsed, but search adv is just around 30%, this means soft bigram + utility reward scaling is hitting a sweet spot already
#     let's stop here on algorithm iteration

# traj_marg_ent = 1.0, bigran zipf reg: 1.0
# traj_loss: 1.51 | abs_loss: 0.46 | search adv: 18.40% | vocab util: 75.00%  | kl_soft_zipf: 0.15 | avg cos sim: 0.50
# degrades search adv, this is not a good idea, drop it

# obs 1. 
# - zipf distribution matching (with kl regularization target) is able to lift up search advantage
# - soft zipf kl seems to work better than hard zipf kl

In [4]:
if len(img_frames) > 0:
    img_frames[0].save(
        'tinystories_dynamics (select-best per abs SoRL + 1.0 bigram zipf reg + utility reward scaling (min 0.5).gif',
        save_all=True,
        append_images=img_frames[1:],
        duration=400,  # milliseconds per frame
        loop=0  # 0 = infinite loop
    )
    print(f"Saved GIF with {len(img_frames)} frames")

In [ ]:
# Issue #1. 
# - the figure is off. 
# - very likely, cluster visualizer has abs # starts from 0, but alignment visualizer has abs # starts from 1.
